In [4]:
import cv2
import numpy as np

def nada(x):
    pass

def iniciar_segmentador(modo="HSV", origen="IMAGEN", ruta_img="futbol.jpg"):
    # Inicializar origen (Webcam o Imagen)
    if origen == "WEBCAM":
        cap = cv2.VideoCapture(0)
        # Darle un segundito a la cámara para que encienda
        if not cap.isOpened():
            print("❌ Error: No se pudo abrir la cámara web.")
            return
    else:
        img = cv2.imread(ruta_img)
        if img is None:
            print(f"❌ Error: No se encontró '{ruta_img}'.")
            return

    # Crear ventana ajustable
    cv2.namedWindow('Segmentador', cv2.WINDOW_NORMAL)
    
    # Trackbar para alternar: 0 = Máscara, 1 = Segmentación
    cv2.createTrackbar('Ver: Masc/Res', 'Segmentador', 1, 1, nada)

    if modo == "HSV":
        cv2.createTrackbar('H Min', 'Segmentador', 0, 179, nada)
        cv2.createTrackbar('S Min', 'Segmentador', 0, 255, nada)
        cv2.createTrackbar('V Min', 'Segmentador', 0, 255, nada)
        cv2.createTrackbar('H Max', 'Segmentador', 179, 179, nada)
        cv2.createTrackbar('S Max', 'Segmentador', 255, 255, nada)
        cv2.createTrackbar('V Max', 'Segmentador', 255, 255, nada)

    print("✅ Interfaz iniciada. Presioná 'q' o cerrá la ventana con la 'X' para salir.")

    while True:
        # ¡LA SOLUCIÓN AL ERROR!: Verificamos si la ventana fue cerrada con la 'X'
        try:
            if cv2.getWindowProperty('Segmentador', cv2.WND_PROP_VISIBLE) < 1:
                break
        except cv2.error:
            break # Si la ventana ya no existe en memoria, salimos del loop

        # Leer cámara o imagen
        if origen == "WEBCAM":
            ret, frame = cap.read()
            if not ret: break
        else:
            frame = img.copy()

        ver_resultado = cv2.getTrackbarPos('Ver: Masc/Res', 'Segmentador')

        # Obtener valores
        if modo == "HSV":
            frame_procesar = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            val_min = np.array([cv2.getTrackbarPos('H Min', 'Segmentador'),
                                cv2.getTrackbarPos('S Min', 'Segmentador'),
                                cv2.getTrackbarPos('V Min', 'Segmentador')])
            val_max = np.array([cv2.getTrackbarPos('H Max', 'Segmentador'),
                                cv2.getTrackbarPos('S Max', 'Segmentador'),
                                cv2.getTrackbarPos('V Max', 'Segmentador')])
            
        # Crear máscara y segmentar
        mascara = cv2.inRange(frame_procesar, val_min, val_max)
        
        if ver_resultado == 1:
            vista_final = cv2.bitwise_and(frame, frame, mask=mascara)
        else:
            vista_final = cv2.cvtColor(mascara, cv2.COLOR_GRAY2BGR)

        # Mostrar lado a lado original y procesado
        cuadro_combinado = np.hstack((frame, vista_final))
        
        # Achicar la ventana para que entre bien en pantallas chicas
        # Calculamos la proporción para que no se deforme
        alto_original, ancho_original = cuadro_combinado.shape[:2]
        nuevo_ancho = 1200
        nuevo_alto = int((nuevo_ancho / ancho_original) * alto_original)
        cuadro_combinado = cv2.resize(cuadro_combinado, (nuevo_ancho, nuevo_alto))
        
        cv2.imshow('Segmentador', cuadro_combinado)

        # Salir con la 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Liberar memoria y apagar cámara
    if origen == "WEBCAM": cap.release()
    cv2.destroyAllWindows()

# PARA PROBAR: Llamamos a la función con tu cámara web
iniciar_segmentador(modo="HSV", origen="WEBCAM")

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1284: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvNamedWindow'


In [5]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntRangeSlider

print("--- EJERCICIO 5.2: Deforestación Interactiva en Tiempo Real ---")

# 1. CARGA Y PROCESAMIENTO INICIAL
img = cv2.imread('Deforestacion.png')

if img is None:
    print("Error: No se encuentra 'Deforestacion.png'.")
else:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Auto-detección del recuadro blanco
    img_gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh_blanco = cv2.threshold(img_gris, 200, 255, cv2.THRESH_BINARY)
    contornos, _ = cv2.findContours(thresh_blanco, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    mascara_roi = np.zeros_like(img_gris)
    if len(contornos) > 0:
        contorno_caja = max(contornos, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contorno_caja)
        if w > 100 and h > 100:
            # Rellenamos el recuadro
            cv2.rectangle(mascara_roi, (x+5, y+5), (x+w-10, y+h-10), 255, thickness=-1)
        else:
            mascara_roi[:] = 255
    else:
        mascara_roi[:] = 255

    # Calculamos la constante de Hectáreas por píxel (basado en 24,500 has)
    pixeles_totales_roi = cv2.countNonZero(mascara_roi)
    has_por_pixel = 24500.0 / pixeles_totales_roi if pixeles_totales_roi > 0 else 0


    # 2. FUNCIÓN INTERACTIVA
    def sintonizar_deforestacion(h_rango, s_rango, v_rango):
        img_resultado = img_rgb.copy()
        
        # Armar rangos desde los sliders
        lower_bound = np.array([h_rango[0], s_rango[0], v_rango[0]])
        upper_bound = np.array([h_rango[1], s_rango[1], v_rango[1]])
        
        # Detectar color en toda la imagen
        mascara_color = cv2.inRange(img_hsv, lower_bound, upper_bound)
        
        # Cruzar con la máscara del recuadro (para ignorar lo de afuera)
        mascara_final = cv2.bitwise_and(mascara_color, mascara_color, mask=mascara_roi)
        
        # Pintar de rojo la zona detectada (Usamos RGB puro: [255, 0, 0] porque mostramos con Matplotlib)
        img_resultado[mascara_final > 0] = [255, 0, 0] 
        
        # Calcular hectáreas en tiempo real
        pixeles_deforestados = cv2.countNonZero(mascara_final)
        area_deforestada = pixeles_deforestados * has_por_pixel
        area_monte = 24500.0 - area_deforestada
        
        # Mostrar resultados
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        
        axs[0].imshow(img_rgb)
        axs[0].set_title("Original")
        axs[0].axis('off')
        
        axs[1].imshow(mascara_final, cmap='gray')
        axs[1].set_title(f"Máscara Detectada\nH:{h_rango} | S:{s_rango} | V:{v_rango}")
        axs[1].axis('off')
        
        axs[2].imshow(img_resultado)
        
        axs[2].set_title(f"Deforestado: {area_deforestada:,.0f} has | Monte: {area_monte:,.0f} has", fontsize=14, fontweight='bold')
        axs[2].axis('off')
        
        plt.tight_layout()
        plt.show()

    # Lanzamos los controles con valores sugeridos para empezar
    interact(sintonizar_deforestacion,
             h_rango=IntRangeSlider(min=0, max=179, value=[10, 60], description='Matiz (H)', continuous_update=False),
             s_rango=IntRangeSlider(min=0, max=255, value=[10, 150], description='Sat (S)', continuous_update=False),
             v_rango=IntRangeSlider(min=0, max=255, value=[60, 220], description='Brillo (V)', continuous_update=False))

--- EJERCICIO 5.2: Deforestación Interactiva en Tiempo Real ---


interactive(children=(IntRangeSlider(value=(10, 60), continuous_update=False, description='Matiz (H)', max=179…